# Jev-CeNN — All 28 attention layers brute-force V4 (FFNs FROZEN)

**Purpose:** Test alternatives in the ATTENTION submodule of every one of the 28 ModernBERT layers, without changing **any** pretrained FFN, LayerNorm, residual connection or task head. Every original encoder layer remains in place. No 3→1 merging, bypasses or teacher FFN reconstruction.

**Two outputs, important distinction:**

1. **Quality-gated + speed-gated student:** scans *every* layer; installs an attention replacement only when cumulative DEV agreement, gold accuracy, cosine and measured GPU speed pass. Rejected layers automatically remain original. This is the recommended output.
2. **Forced all-28 research model:** saves the best-effort trained ATTENTION replacement for *each* layer, including those failing gates, and evaluates the entire forcibly changed encoder only on the independent final HOLDOUT. This explicitly satisfies the all-layer replacement experiment, but is **not assumed accurate** and is not the recommended deployment checkpoint.

**Search:** native global layers get local+global-anchor grids; native local-W129 layers get narrower W33/W65/W97 grids; all options receive a zero-shot scan, top K get cached teacher/prefix distillation. Use `TRAIN_ALL_GRID=True` for full brute-force **training** of every mask (substantially more GPU time). CeNN on/off is evaluated separately. Local targets use the **current student's native layer outputs** to prevent errors from previously changed layers contaminating successive distillation.

**Fidelity guards:** reference attention equivalence per layer, per-layer FFN/LayerNorm SHA-256 checks, cumulative teacher agreement >=99%, gold accuracy drop <=1.5 percentage points, GPU paired A/B/A timing; independent HOLDOUT checked only once after the architecture is finalized.

**Scope of claimed efficiency:** dense SDPA still forms dense masks; narrower windows do NOT guarantee lower FLOPs or better GPU latency. Chunked attention is implemented as an optional exact memory-bounded kernel for local attention. Forced all-28 checkpoints may be larger and slower than the teacher. No unmeasured results are claimed.


In [ ]:
#@title 1. Setup: exhaustive scan over ALL 28 attention layers, never change FFNs
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"

from pathlib import Path
import copy, gc, csv, hashlib, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question, full_attention_indices
)
from tinycenn_lm.decision_training import require_finite
from tinycenn_lm.jev_cenn_full_local_repeat import cpu_state_snapshot
from tinycenn_lm.laya_lab.core import BaseLayaReplacementAttention, _valid_tokens
from tinycenn_lm.jev_cenn_pairwindow_superlayer import CellularMultiscale

SEED=2028
SOURCE_MODEL="convaiinnovations/laya-typed-decisions" #@param {type:"string"}
MAX_LEN=512
LAYER_ORDER=tuple(range(28)) # Every layer is attempted, no groups bypassed.
# Original global: 0,3,6,9,12,15,18,21,24,27. Others native W129.
GLOBAL_GRID=(
 (129,"anchors",16),(129,"anchors",32),
 (257,"anchors",16),(257,"anchors",32),
 (385,"anchors",16),(385,"anchors",32),
 (449,"anchors",16),(481,"anchors",16),
)
LOCAL_GRID=((33,"local",0),(65,"local",0),(97,"local",0))
# Every grid candidate is tested zero-shot. Top candidates get gradient updates.
TRAIN_ALL_GRID=False #@param {type:"boolean"}
TOP_K_TRAIN=2 #@param {type:"integer"}
QUICK_PROBE_BATCHES=3
TRAIN_STEPS_GLOBAL=120 #@param {type:"integer"}
TRAIN_STEPS_LOCAL=80 #@param {type:"integer"}
CACHE_BATCHES_GLOBAL=80
CACHE_BATCHES_LOCAL=48
PROBE_BATCHES=12
BATCH=2
PROBE_EVERY=30
LR_ATTENTION=1.5e-5
LR_CENN=2e-4
COSINE_WEIGHT=3.0
TRAIN_CASES=700
DEV_CASES=220
HOLDOUT_CASES=200
PROBE_CASES=80
DEV_GATE_DECISIONS=240
HOLDOUT_DECISIONS=500
TARGET_COSINE=0.995
TARGET_NMSE=0.025
TARGET_AGREEMENT=0.99
MAX_GOLD_DROP=0.015
MAX_INCREMENTAL_AGREEMENT_DROP=0.01
MIN_SPEEDUP=1.015 # Set <=1.0 to study quality-only replacements.
REQUIRE_SPEEDUP=True #@param {type:"boolean"}
# Forced model: all 28 ATTENTION modules replaced with best-effort learned
# sparse variants even if they fail gates. This is a research stress test ONLY.
EVALUATE_FORCED_ALL=True #@param {type:"boolean"}
BENCH_REPEATS=10
USE_FP16_AMP=False
ENABLE_FLEX=False # avoids variable-length recompilation during search
FLEX_VALIDATE_ATOL=5e-3
OUTPUT_DIR=Path("/content/Jev_CeNN_All28_AttentionOnly_BruteForce_V4")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert len(LAYER_ORDER)==28 and set(LAYER_ORDER)==set(range(28))
assert TOP_K_TRAIN>0 and BATCH>0 and PROBE_BATCHES>=QUICK_PROBE_BATCHES
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU: Runtime > Change runtime type.")
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda")
amp_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
source_dir=Path(snapshot_download(
    SOURCE_MODEL,allow_patterns=[
      "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"]))
teacher,source_cfg=build_checkpoint_model(source_dir)
student_eval,_=build_checkpoint_model(source_dir)
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student_eval.encoder.config.reference_compile=False
GLOBAL_LAYERS=tuple(full_attention_indices(teacher))
assert len(teacher.encoder.layers)==len(student_eval.encoder.layers)==28
assert GLOBAL_LAYERS==tuple(range(0,28,3)),GLOBAL_LAYERS
print("GPU:",torch.cuda.get_device_name(0),"global layers:",GLOBAL_LAYERS)
print("All 28 attention layers will be tested. FFNs and LayerNorms are frozen.")


In [ ]:
#@title 3. Partition dataset by CASE: TRAIN, PROBE, DEV-GATE and sealed HOLDOUT
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

from tinycenn_lm.jev_cenn_selective import (
    verify_case_partition, judge_replacement, plan_active_layers,
)
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
rng=random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES+DEV_CASES+HOLDOUT_CASES<=len(rows), (
    f"Required TRAIN+DEV+HOLDOUT cases exceeds {len(rows)} dataset TRAIN cases")
train_rows=rows[:TRAIN_CASES]
dev_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES]
holdout_rows=rows[TRAIN_CASES+DEV_CASES:TRAIN_CASES+DEV_CASES+HOLDOUT_CASES]
split_info=verify_case_partition(train_rows,dev_rows,holdout_rows)
probe_rows=dev_rows[:PROBE_CASES]
gate_rows=dev_rows[PROBE_CASES:]
verify_case_partition(train_rows,probe_rows,gate_rows)
assert not {str(x["id"]) for x in holdout_rows}&{
    str(x["id"]) for x in dev_rows}
train_items=make_items_from_rows(train_rows,SEED,True)
probe_items=make_items_from_rows(probe_rows,SEED+1,True)
gate_items=make_items_from_rows(gate_rows,SEED+2,True)
# IMPORTANT: holdout_items are NOT constructed yet. The final evaluation
# cell materializes these previously sealed rows after all decisions are made.
assert len(gate_items)>=DEV_GATE_DECISIONS
assert len(probe_items)>=PROBE_BATCHES*BATCH
print("Independent partitions:",split_info,
      "probe cases",len(probe_rows),"gate cases",len(gate_rows))
print("Optimizer/probe/gate decisions:",len(train_items),
      len(probe_items),len(gate_items))

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

# Only case-isolated TRAIN / PROBE / DEV-GATE inputs are available here.
# The independent HOLDOUT remains sealed until the final evaluation cell.
print("TRAIN / PROBE / DEV-GATE decisions:",
      len(train_items), len(probe_items), len(gate_items))
print("Official TEST untouched; independent HOLDOUT not materialized.")


In [ ]:
#@title 3. Correct prefix caching: student-native targets for cumulative distillation
def check_finite(tensor,caption):
    if not torch.isfinite(tensor.float()).all():
        raise FloatingPointError(caption)

def to_cuda(row):
    return {k:v.to(device,non_blocking=True) for k,v in row.items()}

def frozen_fingerprint(model):
    sha=hashlib.sha256()
    for i,layer in enumerate(model.encoder.layers):
        for module_name in ("mlp","mlp_norm","attn_norm"):
            module=getattr(layer,module_name)
            for key,tensor in module.state_dict().items():
                sha.update(f"{i}:{module_name}:{key}".encode())
                sha.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return sha.hexdigest()

FFN_NORM_FINGERPRINT=frozen_fingerprint(student_eval)
def assert_frozen_ffn(model=student_eval):
    actual=frozen_fingerprint(model)
    assert actual==FFN_NORM_FINGERPRINT,(
        "Pretrained FFN or norm weights changed! Model invalid.")
    for layer in model.encoder.layers:
        assert not any(x.requires_grad for x in layer.mlp.parameters())
        assert not any(x.requires_grad for x in layer.mlp_norm.parameters())
        assert not any(x.requires_grad for x in layer.attn_norm.parameters())

def cache_layer(layer_index,items,n_batches,seed):
    chosen=random.Random(seed).sample(items,min(len(items),n_batches*BATCH))
    out=[]
    # Native attention must still be installed at target at cache time.
    assert not isinstance(student_eval.encoder.layers[layer_index].attn,WindowCeNNAttention)
    for at in range(0,len(chosen)-BATCH+1,BATCH):
        selected=chosen[at:at+BATCH]
        inputs=direct_batch(selected)
        teach={};current={}
        def before_teacher(mod,args,kwargs):
            teach["x"]=args[0].detach()
            teach["position_ids"]=kwargs.get("position_ids")
        def after_teacher(mod,args,kwargs,result):
            teach["target"]=result[0].detach()
        def before_student(mod,args,kwargs):
            current["x"]=args[0].detach()
            current["position_ids"]=kwargs.get("position_ids")
        def after_student(mod,args,kwargs,result):
            current["native_target"]=result[0].detach()
        tl=teacher.encoder.layers[layer_index]
        sl=student_eval.encoder.layers[layer_index]
        h1=tl.register_forward_pre_hook(before_teacher,with_kwargs=True)
        h2=tl.register_forward_hook(after_teacher,with_kwargs=True)
        try:
            with torch.inference_mode():teacher(**inputs)
        finally:h1.remove();h2.remove()
        h3=sl.register_forward_pre_hook(before_student,with_kwargs=True)
        h4=sl.register_forward_hook(after_student,with_kwargs=True)
        try:
            with torch.inference_mode():student_eval(**inputs)
        finally:h3.remove();h4.remove()
        assert all(k in teach for k in ("x","position_ids","target"))
        assert all(k in current for k in ("x","position_ids","native_target"))
        row={
            "x":teach["x"],
            "teacher_target":teach["target"],
            "rollout":current["x"],
            "native_target":current["native_target"],
            "valid":inputs["attention_mask"].bool(),
            "position_ids":current["position_ids"],
        }
        out.append({
            k:(v.detach().float().cpu().contiguous()
               if v.is_floating_point() else v.detach().cpu().contiguous())
            for k,v in row.items()})
    return out

def new_layer_caches(layer_index):
    n=CACHE_BATCHES_GLOBAL if layer_index in GLOBAL_LAYERS else CACHE_BATCHES_LOCAL
    train=cache_layer(layer_index,train_items,n,SEED+layer_index*13+1)
    probe=cache_layer(layer_index,probe_items,PROBE_BATCHES,SEED+layer_index*13+2)
    return train,probe

print("SHA-256 of all 28 FFNs + attention/FFN LayerNorms:",
      FFN_NORM_FINGERPRINT)


In [ ]:
#@title 6. Cosine/NMSE and cached independent DEV-GATE teacher predictions
def feature_loss(output,reference,valid):
    pred=output.float()[valid]
    gold=reference.float()[valid]
    # Source activations are validated during cache construction; the
    # differentiable scalar loss is checked once per update. Avoid CUDA
    # sync in these two per-forward tensor checks.
    nmse=(pred-gold).square().sum()/gold.square().sum().clamp_min(1e-8)
    cosine=F.cosine_similarity(pred,gold,dim=-1).mean()
    # Do not call Python float() inside the 520-step CUDA training loop.
    # Converting GPU tensors to floats forces a device synchronization.
    return nmse+COSINE_WEIGHT*(1-cosine),nmse.detach(),cosine.detach()
@torch.inference_mode()
def cached_local_quality(candidate,cache,key="rollout"):
    candidate.eval()
    cs=err=energy=tokens=0.0
    for row in cache:
        e=to_cuda(row)
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            pred=candidate(e[key],attention_mask=e["valid"],
                           position_ids=e["position_ids"])[0]
        a=pred.float()[e["valid"]]
        b=e["end_target"].float()[e["valid"]]
        check_finite(a,"local prediction")
        cs+=float(F.cosine_similarity(a,b,dim=-1).sum())
        err+=float((a-b).square().sum())
        energy+=float(b.square().sum())
        tokens+=int(e["valid"].sum())
    return {"cosine":cs/max(tokens,1),"nmse":err/max(energy,1e-8),
            "tokens":int(tokens)}
def cached_rank(teacher_view,prefix_view):
    return (0.20*(teacher_view["nmse"]+COSINE_WEIGHT*(1-teacher_view["cosine"]))
           +0.80*(prefix_view["nmse"]+COSINE_WEIGHT*(1-prefix_view["cosine"])))
def cache_teacher_predictions(items):
    examples=[]
    for i in range(0,len(items),BATCH):
        inp=direct_batch(items[i:i+BATCH])
        with torch.inference_mode(),torch.autocast(
                "cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            logits,_=teacher(**inp)
        examples.append({
            "batch":inp,
            "teacher":logits.float().detach(),
            "gold":torch.tensor(
                [r["gold_index"] for r in items[i:i+BATCH]],
                device=device,dtype=torch.long)})
    return examples
GATE_ITEMS=gate_items[:DEV_GATE_DECISIONS]
assert len(GATE_ITEMS)==DEV_GATE_DECISIONS
GATE_CACHE=cache_teacher_predictions(GATE_ITEMS)
@torch.inference_mode()
def decision_quality(model,examples):
    model.eval()
    correct=student_gold=teacher_gold=n=0
    js_sum=0.
    for e in examples:
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            logits,_=model(**e["batch"])
        check_finite(logits,"full-model decision logits")
        mask=e["batch"]["marker_mask"].bool()
        a=torch.softmax(e["teacher"].masked_fill(~mask,-1e4),-1)
        b=torch.softmax(logits.float().masked_fill(~mask,-1e4),-1)
        av,bv=a.argmax(-1),b.argmax(-1)
        center=(a+b)/2
        js_sum+=float((0.5*(a*(a.clamp_min(1e-9).log()-
                                  center.clamp_min(1e-9).log())).sum(-1)
                      +0.5*(b*(b.clamp_min(1e-9).log()-
                                  center.clamp_min(1e-9).log())).sum(-1)).sum())
        correct+=int((av==bv).sum())
        student_gold+=int((bv==e["gold"]).sum())
        teacher_gold+=int((av==e["gold"]).sum())
        n+=len(av)
    return {"decisions":n,"agreement":correct/max(n,1),
            "student_gold_accuracy":student_gold/max(n,1),
            "teacher_gold_accuracy":teacher_gold/max(n,1),
            "mean_js":js_sum/max(n,1)}
ORIGINAL_GATE=decision_quality(student_eval,GATE_CACHE)
assert ORIGINAL_GATE["agreement"]==1.
print("Original (no compression) DEV-GATE:",ORIGINAL_GATE)


In [ ]:
#@title 5. Exact ModernBERT attention interface; local / anchor / full masks
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb

def attention_sdpa(q,k,v,valid,window,pattern="local",anchors=0):
    """Functional attention with per-example right-padding-aware global anchors."""
    b,h,t,d=q.shape
    pos=torch.arange(t,device=q.device)
    if pattern=="full" or window>=t:
        allowed=valid[:,None,None,:]
    else:
        radius=(int(window)-1)//2
        local=(pos[:,None]-pos[None,:]).abs()<=radius
        allowed=valid[:,None,None,:]&local[None,None,:,:]
        if pattern=="anchors":
            ends=valid.long().sum(-1).clamp_min(1)
            keys=(pos[None,:]<anchors)|(
                pos[None,:]>=(ends[:,None]-anchors).clamp_min(0))
            half=max(1,anchors//2)
            queries=(pos[None,:]<half)|(
                pos[None,:]>=(ends[:,None]-half).clamp_min(0))
            allowed=valid[:,None,None,:]&(
                allowed|keys[:,None,None,:]|queries[:,None,:,None])
    out=F.scaled_dot_product_attention(
        q,k,v,attn_mask=allowed,dropout_p=0.0,is_causal=False)
    return out*valid[:,None,:,None].to(out.dtype)

def chunked_local_sdpa(q,k,v,valid,window,chunk=64):
    """Bound mask memory to O(T*(W+chunk)), rather than O(T*T)."""
    b,h,t,d=q.shape
    if window>=t:return attention_sdpa(q,k,v,valid,window)
    radius=(window-1)//2
    pieces=[]
    for start in range(0,t,chunk):
        stop=min(start+chunk,t)
        first=max(0,start-radius)
        last=min(t,stop+radius)
        qi=torch.arange(start,stop,device=q.device)
        ki=torch.arange(first,last,device=q.device)
        nearby=(qi[:,None]-ki[None,:]).abs()<=radius
        ok=valid[:,None,None,first:last]&nearby[None,None,:,:]
        self_key=(qi[:,None]==ki[None,:])[None,None,:,:]
        ok=ok|(~ok.any(dim=-1,keepdim=True)&self_key)
        piece=F.scaled_dot_product_attention(
            q[:,:,start:stop],k[:,:,first:last],v[:,:,first:last],
            attn_mask=ok,dropout_p=0.0)
        pieces.append(piece)
    out=torch.cat(pieces,dim=-2)
    return out*valid[:,None,:,None].to(out.dtype)

class WindowCeNNAttention(BaseLayaReplacementAttention):
    architecture="jev_cenn_all28_attention_only_frozen_ffn_v4"
    def __init__(self,original,window,pattern,anchors):
        super().__init__(original)
        self.window=int(window);self.pattern=str(pattern);self.anchors=int(anchors)
        self.rotary_emb=copy.deepcopy(original.rotary_emb)
        self.rotary_emb.requires_grad_(False)
        self.cenn=CellularMultiscale(self.hidden_size)
        self.cenn_gain=nn.Parameter(torch.tensor(0.0))
        self.use_cenn=True
        self.backend="dense"
        self.backend_used=None
        self.Wqkv.requires_grad_(True)
        self.Wo.requires_grad_(True)
        self.out_drop.requires_grad_(False)

    def forward(self,hidden_states,position_ids=None,attention_mask=None,**kwargs):
        b,t,d=hidden_states.shape
        if position_ids is None:
            position_ids=torch.arange(t,device=hidden_states.device)[None,:]
        packed=self.Wqkv(hidden_states).view(
            b,t,3,self.num_heads,self.head_dim)
        q,k,v=packed.transpose(1,3).unbind(2)
        cos,sin=self.rotary_emb(packed,position_ids=position_ids)
        q,k=apply_rotary_pos_emb(q,k,cos,sin)
        valid=_valid_tokens(attention_mask,hidden_states)
        if self.backend=="chunked" and self.pattern=="local":
            result=chunked_local_sdpa(q,k,v,valid,self.window)
            self.backend_used="chunked"
        else:
            result=attention_sdpa(q,k,v,valid,self.window,
                                  self.pattern,self.anchors)
            self.backend_used="dense"
        if self.use_cenn:
            # This pathway can be OFF for free; initial gain = exactly zero.
            flat=result.transpose(1,2).reshape(b,t,d)
            flat=flat+0.1*torch.tanh(self.cenn_gain)*self.cenn(flat,valid)
            result=flat.view(b,t,self.num_heads,self.head_dim).transpose(1,2)
        return self.finish(result,hidden_states)

def make_candidate(layer_idx,config):
    window,pattern,anchors=config
    return WindowCeNNAttention(
        teacher.encoder.layers[layer_idx].attn,
        window,pattern,anchors).to(device)

@torch.inference_mode()
def feature_probe(candidate,layer_idx,rows):
    """Incremental fidelity against the native target on CURRENT student inputs."""
    layer=student_eval.encoder.layers[layer_idx]
    native=layer.attn
    layer.attn=candidate
    ce=err=energy=tokens=0.0
    candidate.eval()
    try:
        for row in rows:
            t=to_cuda(row)
            out=layer(t["rollout"],attention_mask=t["valid"],
                      position_ids=t["position_ids"])[0]
            p=out.float()[t["valid"]]
            ref=t["native_target"].float()[t["valid"]]
            ce+=float(F.cosine_similarity(p,ref,dim=-1).sum())
            err+=float((p-ref).square().sum())
            energy+=float(ref.square().sum())
            tokens+=int(t["valid"].sum())
        return {"cosine":ce/max(tokens,1),"nmse":err/max(energy,1e-9),
                "tokens":int(tokens)}
    finally:layer.attn=native

def local_loss(out,target,valid):
    p=out.float()[valid];ref=target.float()[valid]
    nmse=(p-ref).square().sum()/ref.square().sum().clamp_min(1e-8)
    cos=F.cosine_similarity(p,ref,dim=-1).mean()
    return nmse+COSINE_WEIGHT*(1-cos)

@torch.inference_mode()
def verify_native_attention(layer_idx,rows):
    # Native-equivalent full/local control must reproduce same current-student outputs.
    original=student_eval.encoder.layers[layer_idx].attn
    config=((MAX_LEN,"full",0) if layer_idx in GLOBAL_LAYERS
            else (129,"local",0))
    replica=make_candidate(layer_idx,config)
    replica.use_cenn=False
    q=feature_probe(replica,layer_idx,rows[:2])
    del replica
    if q["cosine"]<0.9999 or q["nmse"]>0.00005:
        raise RuntimeError(f"Layer {layer_idx}: native attention not reproduced: {q}")
    return q

print("Per-layer attention implementation ready; native FFNs untouched.")


In [ ]:
#@title 6. Brute-force masked search, train top candidates, CeNN on/off
@torch.inference_mode()
def zero_shot_scan(layer_idx,configs,probe_rows):
    scanned=[]
    for cfg in configs:
        candidate=make_candidate(layer_idx,cfg).eval()
        candidate.use_cenn=False
        q=feature_probe(candidate,layer_idx,probe_rows[:QUICK_PROBE_BATCHES])
        score=q["nmse"]+COSINE_WEIGHT*(1-q["cosine"])
        scanned.append({"config":list(cfg),"score":score,"local":q})
        del candidate
    scanned.sort(key=lambda x:x["score"])
    return scanned

def train_attention(candidate,layer_idx,train_cache,probe_cache,steps):
    candidate.backend="dense"
    candidate.train().requires_grad_(True)
    candidate.rotary_emb.requires_grad_(False)
    candidate.out_drop.requires_grad_(False)
    params_qkv=list(candidate.Wqkv.parameters())+list(candidate.Wo.parameters())
    params_ce=list(candidate.cenn.parameters())+[candidate.cenn_gain]
    opt=torch.optim.AdamW([
        {"params":params_qkv,"lr":LR_ATTENTION},
        {"params":params_ce,"lr":LR_CENN}],weight_decay=1e-4)
    best_rank=float("inf");best=None;best_step=0
    history=[];indices=list(range(len(train_cache)));index=0
    rng=random.Random(SEED+layer_idx*719+candidate.window)
    layer=student_eval.encoder.layers[layer_idx]
    native=layer.attn
    for step in range(1,steps+1):
        if index==0:rng.shuffle(indices)
        record=to_cuda(train_cache[indices[index]])
        index=(index+1)%len(indices)
        candidate.train()
        opt.zero_grad(set_to_none=True)
        layer.attn=candidate
        try:
            output=layer(record["rollout"],attention_mask=record["valid"],
                         position_ids=record["position_ids"])[0]
            loss=local_loss(output,record["native_target"],record["valid"])
            require_finite(loss,"attention-only loss")
            loss.backward()
        finally:
            layer.attn=native
        torch.nn.utils.clip_grad_norm_(params_qkv+params_ce,0.5)
        opt.step()
        if step==1 or step%PROBE_EVERY==0 or step==steps:
            cand_q=feature_probe(candidate,layer_idx,probe_cache)
            rank=cand_q["nmse"]+COSINE_WEIGHT*(1-cand_q["cosine"])
            history.append({"step":step,"probe":cand_q,"score":rank})
            if rank<best_rank:
                best_rank,best_step,best=rank,step,cpu_state_snapshot(candidate)
            print(f"  L{layer_idx:02d} W{candidate.window} "
                  f"{candidate.pattern} train {step}/{steps} "
                  f"cos={cand_q['cosine']:.6f} nmse={cand_q['nmse']:.5g}")
    assert best is not None
    candidate.load_state_dict(best,strict=True)
    candidate.eval().requires_grad_(False)
    del opt,best
    return {"best_step":best_step,"steps":steps,"history":history}

def gate_candidate(candidate,layer_idx,probe_rows,original_gate):
    """Always restore target attention afterwards, including on exceptions."""
    layer=student_eval.encoder.layers[layer_idx]
    native=layer.attn
    layer.attn=candidate
    try:
        q=decision_quality(student_eval,GATE_CACHE)
    finally:
        layer.attn=native
    local=feature_probe(candidate,layer_idx,probe_rows)
    verdict=judge_replacement(
        original_gate,q,local_cosine=local["cosine"],local_nmse=local["nmse"],
        min_agreement=TARGET_AGREEMENT,
        max_incremental_agreement_drop=MAX_INCREMENTAL_AGREEMENT_DROP,
        max_gold_drop=MAX_GOLD_DROP,min_local_cosine=TARGET_COSINE,
        max_local_nmse=TARGET_NMSE,require_speedup=False)
    return {"quality":q,"local":local,"gate":verdict,
            "quality_accepted":bool(verdict["accepted"])}


In [ ]:
#@title 7. GPU timing: paired end-to-end, B1/B2 and actual short/long
@torch.inference_mode()
def cuda_latency(model,inputs,repeats=BENCH_REPEATS):
    model.eval()
    for _ in range(4):model(**inputs)
    torch.cuda.synchronize()
    ms=[]
    for _ in range(repeats):
        start=torch.cuda.Event(enable_timing=True)
        stop=torch.cuda.Event(enable_timing=True)
        start.record();model(**inputs);stop.record()
        stop.synchronize();ms.append(float(start.elapsed_time(stop)))
    return {"median_ms":float(np.median(ms)),"p95_ms":float(np.percentile(ms,95)),
            "samples":repeats}

def sample_benchmark_items():
    # Real padded tensors (no invented data); paired timings on identical input.
    examples=[GATE_ITEMS[0]]
    if len(GATE_ITEMS)>1:examples.append(GATE_ITEMS[1])
    return [direct_batch(examples[:1]),direct_batch(examples[:2])]

BENCH_INPUTS=sample_benchmark_items()
@torch.inference_mode()
def paired_candidate_latency(candidate,layer_idx,inp,repeats=BENCH_REPEATS):
    layer=student_eval.encoder.layers[layer_idx]
    native=layer.attn
    try:
        layer.attn=native
        a=cuda_latency(student_eval,inp,repeats)
        layer.attn=candidate
        b=cuda_latency(student_eval,inp,repeats)
        layer.attn=native
        c=cuda_latency(student_eval,inp,max(4,repeats//2))
        ref=(a["median_ms"]+c["median_ms"])/2.
        return {"before_ms":ref,"after_ms":b["median_ms"],
                "speedup":ref/max(b["median_ms"],1e-9),
                "native_first":a,"candidate":b,"native_second":c}
    finally:layer.attn=native

def actual_length(row):
    return len(row["ids"])

def benchmark_model_pairs(baseline,alternative,items):
    results=[]
    groups={
        "short":[x for x in items if len(x["ids"])<=160],
        "long":[x for x in items if 384<=len(x["ids"])<=MAX_LEN],
    }
    for tag,found in groups.items():
        for n in (1,2):
            if len(found)<n:
                results.append({"length_type":tag,"batch_size":n,
                                "status":"no_real_samples"})
                continue
            chosen=found[:n]
            batch=direct_batch(chosen)
            a=cuda_latency(baseline,batch)
            b=cuda_latency(alternative,batch)
            results.append({"length_type":tag,"batch_size":n,
                            "lengths":[len(x["ids"]) for x in chosen],
                            "native":a,"student":b,
                            "speedup":a["median_ms"]/max(b["median_ms"],1e-9),
                            "status":"measured"})
    return results


In [ ]:
#@title 8. ALL-28 search: every layer scanned; frozen FFNs; per-layer rollback
ALL_LAYER_TRIALS=[]
ACCEPTED_LAYERS=[]
ATTEMPT_FILES={}
INITIAL_GATE=decision_quality(student_eval,GATE_CACHE)
assert INITIAL_GATE["agreement"]==1.0
for layer_idx in LAYER_ORDER:
    print("\n"+"="*70)
    print(f"LAYER {layer_idx}/27 | accepted so far: {ACCEPTED_LAYERS}")
    print("="*70)
    start=time.perf_counter()
    current=decision_quality(student_eval,GATE_CACHE)
    train_cache,probe_cache=new_layer_caches(layer_idx)
    reference=verify_native_attention(layer_idx,probe_cache)
    grid=GLOBAL_GRID if layer_idx in GLOBAL_LAYERS else LOCAL_GRID
    scanned=zero_shot_scan(layer_idx,grid,probe_cache)
    shortlisted=(scanned if TRAIN_ALL_GRID else scanned[:TOP_K_TRAIN])
    tested=[];best_attempt=None;best_attempt_rank=None
    successful=[]
    n_steps=(TRAIN_STEPS_GLOBAL if layer_idx in GLOBAL_LAYERS
             else TRAIN_STEPS_LOCAL)
    for shortlist in shortlisted:
        cfg=tuple(shortlist["config"])
        candidate=make_candidate(layer_idx,cfg)
        fitted=train_attention(candidate,layer_idx,train_cache,probe_cache,n_steps)
        for cenn_on in (False,True):
            candidate.use_cenn=cenn_on
            metrics=gate_candidate(candidate,layer_idx,probe_cache,current)
            result={
                "layer":layer_idx,"config":list(cfg),"cenn_on":cenn_on,
                "fit":fitted,"before":current,
                **metrics}
            tested.append(result)
            # Best effort forced-all ATTENTION checkpoint ranked by teacher
            # decisions, not only cosine; never consult final HOLDOUT.
            attempt_rank=(metrics["quality"]["agreement"],
                          metrics["quality"]["student_gold_accuracy"],
                          metrics["local"]["cosine"])
            if best_attempt_rank is None or attempt_rank>best_attempt_rank:
                best_attempt_rank=attempt_rank
                best_attempt=(cfg,cenn_on,cpu_state_snapshot(candidate))
            if metrics["quality_accepted"]:
                # Benchmark only quality-eligible candidates: expensive E2E
                candidate.eval().requires_grad_(False)
                bench=paired_candidate_latency(candidate,layer_idx,
                                              BENCH_INPUTS[0],BENCH_REPEATS)
                result["speed"]=bench
                result["speed_accepted"]=(bench["speedup"]>=MIN_SPEEDUP
                    or not REQUIRE_SPEEDUP)
                if result["speed_accepted"]:
                    successful.append((result,cpu_state_snapshot(candidate)))
            print(f"  cfg={cfg} CeNN={cenn_on}: "
                  f"agreement={metrics['quality']['agreement']:.3%} "
                  f"cos={metrics['local']['cosine']:.6f} "
                  f"gate={metrics['quality_accepted']}")
        del candidate
        gc.collect()
    # ALWAYS store attempted best-per-layer even if the native option wins.
    assert best_attempt is not None
    best_cfg,best_cenn,best_state=best_attempt
    best_path=OUTPUT_DIR/f"forced_best_attention_L{layer_idx:02d}.pt"
    torch.save({"layer":layer_idx,"config":list(best_cfg),
                "cenn_on":best_cenn,"state":best_state,
                "quality_gate_not_required":True},best_path)
    ATTEMPT_FILES[layer_idx]=str(best_path)
    del best_state,best_attempt
    selected=None
    if successful:
        # Rank strictly among candidates satisfying BOTH quality and speed.
        successful.sort(key=lambda x:(
            x[0]["speed"]["speedup"],
            x[0]["quality"]["agreement"],
            x[0]["local"]["cosine"]),reverse=True)
        winning,weights=successful[0]
        cfg=tuple(winning["config"])
        candidate=make_candidate(layer_idx,cfg)
        candidate.load_state_dict(weights,strict=True)
        candidate.use_cenn=winning["cenn_on"]
        candidate.eval().requires_grad_(False)
        native=student_eval.encoder.layers[layer_idx].attn
        student_eval.encoder.layers[layer_idx].attn=candidate
        after=decision_quality(student_eval,GATE_CACHE)
        # Final gate computed after actual installation to guard state glitches.
        if after["agreement"]<TARGET_AGREEMENT or (
           after["student_gold_accuracy"]<
           current["student_gold_accuracy"]-MAX_GOLD_DROP):
            student_eval.encoder.layers[layer_idx].attn=native
            print("  ROLLBACK: selected installation failed cumulative gate")
        else:
            selected={
                "config":list(cfg),"cenn_on":winning["cenn_on"],
                "quality":after,"speed":winning["speed"]}
            ACCEPTED_LAYERS.append(layer_idx)
            torch.save({
                "layer":layer_idx,"config":list(cfg),
                "cenn_on":winning["cenn_on"],
                "state":cpu_state_snapshot(candidate),
                "gate":winning["gate"],"speed":winning["speed"]},
                OUTPUT_DIR/f"accepted_attention_L{layer_idx:02d}.pt")
            print("  ACCEPTED",selected)
    del successful
    assert_frozen_ffn()
    ALL_LAYER_TRIALS.append({
        "layer":layer_idx,"native_pattern":
            "global" if layer_idx in GLOBAL_LAYERS else "local129",
        "reference":reference,"zero_shot_grid":scanned,
        "trained_candidates":tested,"selected":selected,
        "current_quality_after":decision_quality(student_eval,GATE_CACHE),
        "elapsed_seconds":time.perf_counter()-start,
        "best_effort_checkpoint":str(best_path),
    })
    (OUTPUT_DIR/"all28_progress.json").write_text(
        json.dumps({
            "processed_layers":[row["layer"] for row in ALL_LAYER_TRIALS],
            "accepted_layers":ACCEPTED_LAYERS,
            "trials":ALL_LAYER_TRIALS,
            "current_gate":ALL_LAYER_TRIALS[-1]["current_quality_after"]
        },indent=2,allow_nan=False))
    del train_cache,probe_cache
    gc.collect()
    torch.cuda.empty_cache()
print("ALL 28 encoder attention layers evaluated.")
print("Quality+speed accepted:",ACCEPTED_LAYERS)
assert_frozen_ffn()


In [ ]:
#@title 9. Final sealed HOLDOUT, forced all-28 stress test, reports and ZIP export
# HOLDOUT is created only now, AFTER choosing the entire architecture.
holdout_items=make_items_from_rows(holdout_rows,SEED+999,False)
assert len(holdout_items)>=HOLDOUT_DECISIONS
HOLDOUT_ITEMS=holdout_items[:HOLDOUT_DECISIONS]
HOLDOUT_CACHE=cache_teacher_predictions(HOLDOUT_ITEMS)
SAFE_GATE=decision_quality(student_eval,GATE_CACHE)
SAFE_HOLDOUT=decision_quality(student_eval,HOLDOUT_CACHE)
SAFE_SPEED=benchmark_model_pairs(teacher,student_eval,GATE_ITEMS+probe_items)
assert_frozen_ffn()

# A distinct, explicitly UNSAFE brute-force output that changes attention in
# EVERY layer regardless of gates. Never overwrite the quality-gated student.
FORCED_REPORT={"evaluated":False,"warning":
    "Best-effort all-layer attention replacement; NOT automatically accepted"}
if EVALUATE_FORCED_ALL:
    # Reduce simultaneous model GPU memory. Safe model results already stored.
    student_eval.cpu()
    torch.cuda.empty_cache()
    brute_model,_=build_checkpoint_model(source_dir)
    brute_model.eval().requires_grad_(False)
    brute_model.encoder.config.reference_compile=False
    for idx in LAYER_ORDER:
        saved=torch.load(ATTEMPT_FILES[idx],map_location="cpu",weights_only=False)
        original=brute_model.encoder.layers[idx].attn
        window,pattern,anchors=saved["config"]
        module=WindowCeNNAttention(original,window,pattern,anchors)
        module.load_state_dict(saved["state"],strict=True)
        module.use_cenn=bool(saved["cenn_on"])
        module.eval().requires_grad_(False)
        brute_model.encoder.layers[idx].attn=module
    assert_frozen_ffn(brute_model)
    brute_model.to(device)
    with torch.inference_mode():
        forced_gate=decision_quality(brute_model,GATE_CACHE)
        forced_holdout=decision_quality(brute_model,HOLDOUT_CACHE)
        forced_speed=benchmark_model_pairs(teacher,brute_model,
                                           GATE_ITEMS+probe_items)
    FORCED_REPORT={
        "evaluated":True,"attention_layers_replaced":list(LAYER_ORDER),
        "all_original_ffn_and_norms_preserved":True,
        "dev":forced_gate,"holdout":forced_holdout,"speed":forced_speed,
        "warning":"Ungated research stress test: not safe for deployment."}
    print("FORCED ALL-28 holdout agreement:",forced_holdout["agreement"])
    del brute_model
    gc.collect();torch.cuda.empty_cache()
    student_eval.to(device)

native_parameters=sum(x.numel() for x in teacher.parameters())
safe_parameters=sum(x.numel() for x in student_eval.parameters())
report={
    "architecture":"all28_attention_only_ffn_frozen_bruteforce_v4",
    "model_id":SOURCE_MODEL,"seed":SEED,"torch":torch.__version__,
    "gpu":torch.cuda.get_device_name(0),
    "original_encoder_layers":28,"tested_attention_layers":list(LAYER_ORDER),
    "quality_speed_accepted_attention_layers":ACCEPTED_LAYERS,
    "all_ffn_norms_frozen_sha256":FFN_NORM_FINGERPRINT,
    "safe_model_gate":SAFE_GATE,"safe_model_holdout":SAFE_HOLDOUT,
    "safe_model_short_long_speed":SAFE_SPEED,
    "forced_all28_research_model":FORCED_REPORT,
    "original_parameter_count":native_parameters,
    "safe_parameter_count":safe_parameters,
    "safe_parameter_delta":safe_parameters-native_parameters,
    "trials":ALL_LAYER_TRIALS,
    "split":split_info,
    "limitations":[
      "All 28 attention modules were scanned, but only top-k masks were trained unless TRAIN_ALL_GRID=True.",
      "The research forced-all model replaces all attentions regardless of failed quality gates.",
      "All FFNs, FFN norms and attention norms retain exact pretrained tensors.",
      "Attention replacement need not reduce parameter count or GPU latency.",
      "A dense attention mask does not reduce SDPA asymptotic computation.",
      "Per-layer speedup of only ~1% can be GPU timing noise.",
      "Sealed holdout is case-disjoint, but not an independent newly collected dataset.",
      "Official test set was not used.",
      "Torch/CUDA kernels vary by GPU and sequence length."]}
(OUTPUT_DIR/"all28_final_report.json").write_text(
    json.dumps(report,indent=2,allow_nan=False))
with (OUTPUT_DIR/"all28_summary.csv").open("w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=[
        "layer","native_pattern","selected","window","pattern","anchors",
        "cenn_enabled","agreement","gold_accuracy","speedup","elapsed_seconds"])
    w.writeheader()
    for result in ALL_LAYER_TRIALS:
        pick=result["selected"]
        q=pick["quality"] if pick else result["current_quality_after"]
        cfg=pick["config"] if pick else (None,None,None)
        w.writerow({
            "layer":result["layer"],"native_pattern":result["native_pattern"],
            "selected":bool(pick),"window":cfg[0],"pattern":cfg[1],
            "anchors":cfg[2],
            "cenn_enabled":pick["cenn_on"] if pick else None,
            "agreement":q["agreement"],
            "gold_accuracy":q["student_gold_accuracy"],
            "speedup":pick["speed"]["speedup"] if pick else None,
            "elapsed_seconds":result["elapsed_seconds"]})
import zipfile
archive=OUTPUT_DIR/"all28_results.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT_DIR.iterdir()):
        if p.is_file() and p!=archive:z.write(p,p.name)
print(json.dumps({
    "tested":len(ALL_LAYER_TRIALS),"quality_speed_accepted":ACCEPTED_LAYERS,
    "safe_holdout_agreement":SAFE_HOLDOUT["agreement"],
    "safe_holdout_gold":SAFE_HOLDOUT["student_gold_accuracy"],
    "forced_holdout_agreement":FORCED_REPORT.get("holdout",{}).get("agreement"),
    "report":str(OUTPUT_DIR/"all28_final_report.json"),
    "archive":str(archive)},indent=2))
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass
